# faiss
## cpu
* flat
* cluster
* HNSW

## gpu

In [ ]:
import numpy as np
import pandas as pd
import faiss
from datetime import datetime
from tqdm import tqdm

import benchmark_helper

import importlib
importlib.reload(benchmark_helper)


In [3]:
print(faiss.__version__)

1.10.0


In [4]:
# load data
# load sabrina embedding pickle 
path = "/mnt/nas2/sabrina/face-gen/embeddings_sabrina.pkl"

df = pd.read_pickle(path)

is_array_col = df['embedding'].apply(lambda x: isinstance(x, np.ndarray))

# convert back to float32
df['embedding'] = df['embedding'].apply(lambda x: x.astype(np.float32))
print(df.shape)


(13199, 10)


In [11]:
flat_search = faiss_search(df, top_k=10, index_type='flat')


FAISS-FLAT Exclude-Self Search:   0%|          | 0/13199 [00:00<?, ?it/s]

FAISS-FLAT Exclude-Self Search: 100%|██████████| 13199/13199 [04:53<00:00, 45.00it/s] 

Total queries evaluated: 9137
Top-1 Accuracy: 0.9775
Top-3 Accuracy: 0.9845
Top-10 Accuracy: 0.9852
Top-1 Failures: 206
Top-3 Failures: 142
Top-10 Failures: 135
Median Query Time: 0.0029 seconds


In [12]:
ivf_search = faiss_search(df, top_k=10, index_type='ivf')


FAISS-IVF Exclude-Self Search: 100%|██████████| 13199/13199 [46:26<00:00,  4.74it/s] 

Total queries evaluated: 9137
Top-1 Accuracy: 0.8112
Top-3 Accuracy: 0.8164
Top-10 Accuracy: 0.8165
Top-1 Failures: 1725
Top-3 Failures: 1678
Top-10 Failures: 1677
Median Query Time: 0.0002 seconds


In [5]:
hsnw_search = faiss_search(df, top_k=10, index_type='hnsw')


FAISS-HNSW Exclude-Self Search:   0%|          | 0/13199 [00:00<?, ?it/s]

FAISS-HNSW Exclude-Self Search: 100%|██████████| 13199/13199 [4:37:22<00:00,  1.26s/it]  

Total queries evaluated: 9137
Top-1 Accuracy: 0.8867
Top-3 Accuracy: 0.8930
Top-10 Accuracy: 0.8934
Top-1 Failures: 1035
Top-3 Failures: 978
Top-10 Failures: 974
Median Query Time: 0.0002 seconds


# screenshot of the benchmark results

- model name: ResNet50@WebFace600K
- dataset: LFW with Sabrina (13199 images)

| algo | top 1 accuracy | latency (secs) | params |
| ------ |------ |------ | ------ |
| FVS | 97.72% | 0.03011 | n/a |
| hnswlib | 93.26%* | 0.217652* | efc=200, m=16, efs=200 |
| brute force | 97.76% | 0.215531 | n/a |
| FAISS flat | 97.75% | 0.0029 | n/a |
| FAISS IVF | 81.12% | 0.0002 | nlist=100 |
| FAISS HNSW | 88.67% | 0.0002 | m=32 |

*could change based on hyperparameters (efc=200, m=16, efs=200)



In [9]:
benchmark_helper.faiss_search(df, top_k=10, index_type='flat', test_mode=True)

test mode...


FAISS-FLAT Exclude-Self Search:   0%|          | 0/13199 [00:00<?, ?it/s]

FAISS-FLAT Exclude-Self Search:   1%|          | 100/13199 [00:01<03:22, 64.57it/s]

Total queries evaluated: 65
Top-1 Accuracy: 0.9692
Top-3 Accuracy: 1.0000
Top-10 Accuracy: 1.0000
Top-1 Failures: 2
Top-3 Failures: 0
Top-10 Failures: 0
Median Query Time: 0.0024 seconds


{'top1_acc': 0.9692307692307692,
 'top3_acc': 1.0,
 'top10_acc': 1.0,
 'median_latency': np.float64(0.002387),
 'top1_failures': [{'query_idx': 34,
   'query_id': np.int64(20),
   'query_name': 'Abdullah_Gul',
   'top_k_ids': [4551]},
  {'query_idx': 40,
   'query_id': np.int64(20),
   'query_name': 'Abdullah_Gul',
   'top_k_ids': [4551]}],
 'top3_failures': [],
 'top10_failures': []}

## gpu

In [9]:

def faiss_search_gpu(df, top_k=10, index_type='flat', nlist=100, nprobe=10):
    embedding_matrix = np.stack(df['embedding'].values).astype(np.float32)
    dim = embedding_matrix.shape[1]
    gpu_res = faiss.StandardGpuResources()
    index = None

    # --- Build CPU base index and transfer to GPU ---
    if index_type == 'flat':
        base_index = faiss.IndexFlatL2(dim)
        index = faiss.IndexIDMap(base_index)
        index = faiss.index_cpu_to_gpu(gpu_res, 0, index)

    elif index_type == 'ivf':
        quantizer = faiss.IndexFlatL2(dim)
        base_index = faiss.IndexIVFFlat(quantizer, dim, nlist)
        if not base_index.is_trained:
            base_index.train(embedding_matrix)
        base_index.nprobe = nprobe
        index = faiss.IndexIDMap(base_index)
        index = faiss.index_cpu_to_gpu(gpu_res, 0, index)

    else:
        raise ValueError("index_type must be 'flat' or 'ivf'")

    # add embeddings with IDs
    index.add_with_ids(embedding_matrix, np.arange(len(embedding_matrix)))

    top_1_pos, top_3_pos, top_10_pos = 0, 0, 0
    top_1_failures, top_3_failures, top_10_failures = [], [], []
    search_time = []
    total = 0

    for query_idx in tqdm(range(len(df)), desc=f"FAISS-GPU-{index_type.upper()} Exclude-Self"):
        if query_idx >= 100:
            break

        query_vector = embedding_matrix[query_idx].reshape(1, -1)
        query_person_id = df.iloc[query_idx]['ID']
        query_name = df.iloc[query_idx]['name']

        if df[df.ID == query_person_id].shape[0] == 1:
            continue

        total += 1

        # remove query from df
        index.remove_ids(np.array([query_idx], dtype=np.int64))

        start_time = datetime.now()
        _, retrieved_ids = index.search(query_vector, top_k + 5)

        filtered_ids = [i for i in retrieved_ids[0] if i != query_idx][:top_k]
        top_k_ids = df.iloc[filtered_ids]['ID'].tolist()

        # top k
        if top_k_ids and top_k_ids[0] == query_person_id:
            top_1_pos += 1
        else:
            top_1_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_ids': top_k_ids[:1]
            })

        end_time = datetime.now()
        search_time.append((end_time - start_time).total_seconds())

        if query_person_id in top_k_ids[:3]:
            top_3_pos += 1
        else:
            top_3_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_ids': top_k_ids[:3]
            })

        if query_person_id in top_k_ids[:10]:
            top_10_pos += 1
        else:
            top_10_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_ids': top_k_ids
            })

        # Re-add query back to the GPU index
        index.add_with_ids(query_vector, np.array([query_idx], dtype=np.int64))

    # Report
    top1_acc = top_1_pos / total
    top3_acc = top_3_pos / total
    top10_acc = top_10_pos / total
    median_latency = np.median(search_time)

    print(f"Total queries evaluated: {total}")
    print(f"Top-1 Accuracy: {top1_acc:.4f}")
    print(f"Top-3 Accuracy: {top3_acc:.4f}")
    print(f"Top-10 Accuracy: {top10_acc:.4f}")
    print(f"Top-1 Failures: {len(top_1_failures)}")
    print(f"Top-3 Failures: {len(top_3_failures)}")
    print(f"Top-10 Failures: {len(top_10_failures)}")
    print(f"Median Query Time: {median_latency:.4f} seconds")

    return {
        'top1_acc': top1_acc,
        'top3_acc': top3_acc,
        'top10_acc': top10_acc,
        'median_latency': median_latency,
        'top1_failures': top_1_failures,
        'top3_failures': top_3_failures,
        'top10_failures': top_10_failures
    }

In [10]:
results = faiss_search_gpu(df, top_k=10, index_type='flat')


AttributeError: module 'faiss' has no attribute 'StandardGpuResources'

In [ ]:
# or
results = faiss_search_gpu(df, top_k=10, index_type='ivf', nlist=256, nprobe=16)